# Ensemble V30 修正版評価・分析

CMI評価関数の修正後に、詳細な評価と分析を行います。

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple, Any
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow import keras

# プロジェクトのルートディレクトリを追加
import sys
sys.path.append('..')

from src.utils.cmi_evaluation import calculate_cmi_score
from src.utils.pipeline import Preprocessor
from src.trainers.multimodal_trainer_v30 import MultimodalTrainerV30

# 日本語フォント設定
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print("✅ ライブラリ読み込み完了")

## 設定とデータ読み込み

In [ ]:
# 設定
EXPERIMENT_NAME = "20250717_preproc_train_v30"
TRAINER_NAME = "multimodal_v30"
N_FOLDS = 5
RANDOM_SEED = 42

# パス設定
BASE_DIR = Path("../output/experiments")
DATA_DIR = BASE_DIR / EXPERIMENT_NAME / "preprocessed"
MODEL_DIR = BASE_DIR / TRAINER_NAME / "models"
RESULT_DIR = BASE_DIR / TRAINER_NAME / "results"

print(f"実験名: {EXPERIMENT_NAME}")
print(f"データディレクトリ: {DATA_DIR}")
print(f"モデルディレクトリ: {MODEL_DIR}")
print(f"結果ディレクトリ: {RESULT_DIR}")

In [ ]:
# データ読み込み関数
def load_data() -> Dict[str, np.ndarray]:
    """前処理済みデータを読み込む"""
    print("📊 データ読み込み中...")
    
    def _load(name: str) -> np.ndarray:
        npy = DATA_DIR / f"{name}.npy"
        pkl = DATA_DIR / f"{name}.pkl"
        if npy.exists():
            return np.load(npy)
        if pkl.exists():
            with open(pkl, "rb") as f:
                return pickle.load(f)
        raise FileNotFoundError(f"{name} ファイルが見つかりません")
    
    X_sensor = _load("train_windows")
    X_demo = _load("train_demographics")
    X_tab = _load("train_tabular")
    X_tof = _load("train_tof_windows")
    y = _load("train_labels")
    info = _load("train_info")
    
    # グループ情報の抽出
    if isinstance(info, list) and len(info) > 0 and isinstance(info[0], dict):
        groups = np.array([
            d.get("subject") if d.get("subject") is not None else d.get("sequence_id")
            for d in info
        ])
    else:
        groups = np.asarray(info)
    
    print(f"センサー: {X_sensor.shape}")
    print(f"人口統計: {X_demo.shape}")
    print(f"表形式: {X_tab.shape}")
    print(f"ToF: {X_tof.shape}")
    print(f"ラベル: {y.shape}")
    print(f"グループ数: {len(groups)}")
    print(f"ユニークラベル: {np.unique(y)}")
    
    return {
        "sensor": X_sensor,
        "demographics": X_demo,
        "tabular": X_tab,
        "tof": X_tof,
        "labels": y,
        "groups": groups,
    }

# データ読み込み
data = load_data()

## データ基本分析

In [ ]:
# 基本統計
print("📈 データ基本統計:")
print(f"総サンプル数: {len(data['labels'])}")
print(f"クラス数: {len(np.unique(data['labels']))}")
print(f"ユニークグループ数: {len(np.unique(data['groups']))}")

# クラス分布
unique_labels, counts = np.unique(data['labels'], return_counts=True)
print(f"\nクラス分布:")
for label, count in zip(unique_labels, counts):
    print(f"  クラス {label}: {count}サンプル ({count/len(data['labels'])*100:.1f}%)")

# データ型と形状
print(f"\nデータ形状:")
print(f"  センサー: {data['sensor'].shape}, dtype: {data['sensor'].dtype}")
print(f"  人口統計: {data['demographics'].shape}, dtype: {data['demographics'].dtype}")
print(f"  表形式: {data['tabular'].shape}, dtype: {data['tabular'].dtype}")
print(f"  ToF: {data['tof'].shape}, dtype: {data['tof'].dtype}")

# クラス不均衡の計算
imbalance_ratio = counts.max() / counts.min()
print(f"\nクラス不均衡比: {imbalance_ratio:.2f}")
print(f"最大クラス: {counts.max()}サンプル")
print(f"最小クラス: {counts.min()}サンプル")

## 保存済みモデルの確認

In [ ]:
# 保存済みモデルの確認
print("📁 保存済みモデルの確認:")
if MODEL_DIR.exists():
    model_files = list(MODEL_DIR.glob("*.keras"))
    print(f"見つかったモデル数: {len(model_files)}")
    for model_file in model_files:
        print(f"  - {model_file.name}")
else:
    print("❌ モデルディレクトリが見つかりません")

# 結果ファイルの確認
print("\n📊 結果ファイルの確認:")
if RESULT_DIR.exists():
    result_files = list(RESULT_DIR.glob("*.json"))
    print(f"見つかった結果ファイル数: {len(result_files)}")
    for result_file in result_files:
        print(f"  - {result_file.name}")
else:
    print("❌ 結果ディレクトリが見つかりません")

## 各Foldでの評価（修正版）

In [ ]:
# 各foldでの評価を実行（修正版）
def evaluate_fold_fixed(fold: int, model_path: Path, data: Dict[str, np.ndarray]) -> Dict[str, Any]:
    """単一foldのモデルを評価（修正版）"""
    print(f"\n🔄 Fold {fold} の評価中...")
    
    try:
        # モデル読み込み
        model = keras.models.load_model(model_path)
        print(f"✅ モデル読み込み完了: {model_path.name}")
        
        # 同じfold分割を再現
        skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
        splits = list(skf.split(np.arange(len(data["labels"])), data["labels"], data["groups"]))
        train_idx, val_idx = splits[fold - 1]  # foldは1-indexed
        
        # 検証データでの予測
        X_s_val = data["sensor"][val_idx]
        X_d_val = data["demographics"][val_idx]
        X_t_val = data["tabular"][val_idx]
        X_f_val = data["tof"][val_idx]
        y_val = data["labels"][val_idx]
        
        # 予測実行
        preds = model.predict([X_s_val, X_d_val, X_t_val, X_f_val], verbose=0)
        pred_probs = preds
        pred_labels = preds.argmax(axis=1)
        
        # 評価指標計算
        f1_macro = f1_score(y_val, pred_labels, average="macro")
        f1_weighted = f1_score(y_val, pred_labels, average="weighted")
        
        # CMI-score計算（修正版）
        cmi_score, binary_f1, macro_f1, test_accuracy = calculate_cmi_score(pred_labels, y_val)
        
        # 分類レポート
        report = classification_report(y_val, pred_labels, output_dict=True)
        
        results = {
            "fold": fold,
            "val_indices": val_idx.tolist(),
            "predictions": pred_labels.tolist(),
            "prediction_probs": pred_probs.tolist(),
            "true_labels": y_val.tolist(),
            "f1_macro": float(f1_macro),
            "f1_weighted": float(f1_weighted),
            "cmi_score": float(cmi_score),
            "binary_f1": float(binary_f1),
            "macro_f1": float(macro_f1),
            "test_accuracy": float(test_accuracy),
            "classification_report": report,
        }
        
        print(f"Fold {fold} 結果:")
        print(f"  F1 Macro: {f1_macro:.4f}")
        print(f"  CMI Score: {cmi_score:.4f}")
        print(f"  Accuracy: {test_accuracy:.4f}")
        
        return results
        
    except Exception as e:
        print(f"❌ Fold {fold} の評価でエラー: {e}")
        return None

# 全foldの評価実行
fold_results = []
all_predictions = {}
all_prediction_probs = {}

for fold in range(1, N_FOLDS + 1):
    model_path = MODEL_DIR / f"multimodal_model_v30_fold{fold}.keras"
    
    if model_path.exists():
        results = evaluate_fold_fixed(fold, model_path, data)
        if results is not None:
            fold_results.append(results)
            
            # 予測結果を保存
            for idx, pred in zip(results["val_indices"], results["predictions"]):
                all_predictions[idx] = pred
            for idx, pred_prob in zip(results["val_indices"], results["prediction_probs"]):
                all_prediction_probs[idx] = pred_prob
    else:
        print(f"❌ Fold {fold} のモデルが見つかりません: {model_path}")

print(f"\n✅ {len(fold_results)} foldの評価完了")

## アンサンブル予測の生成

In [ ]:
# アンサンブル予測の生成
def generate_ensemble_predictions_fixed(all_prediction_probs: Dict[int, List[float]]) -> Tuple[np.ndarray, np.ndarray]:
    """各foldの予測確率を平均してアンサンブル予測を生成（修正版）"""
    print("\n🎯 アンサンブル予測生成中...")
    
    # 全サンプルのインデックスを取得
    all_indices = sorted(all_prediction_probs.keys())
    
    # 各サンプルについて、全foldの予測確率を平均
    ensemble_probs = []
    ensemble_labels = []
    
    for idx in all_indices:
        # このサンプルの予測確率を収集
        sample_probs = []
        for fold_result in fold_results:
            if idx in fold_result["val_indices"]:
                val_idx_in_fold = fold_result["val_indices"].index(idx)
                sample_probs.append(fold_result["prediction_probs"][val_idx_in_fold])
        
        # 平均を計算
        if sample_probs:
            avg_prob = np.mean(sample_probs, axis=0)
            ensemble_probs.append(avg_prob)
            ensemble_labels.append(avg_prob.argmax())
        else:
            print(f"⚠️  サンプル {idx} の予測が見つかりません")
    
    return np.array(ensemble_probs), np.array(ensemble_labels)

# アンサンブル予測生成
if fold_results:
    ensemble_probs, ensemble_labels = generate_ensemble_predictions_fixed(all_prediction_probs)
    
    # アンサンブル予測の評価
    all_indices = sorted(all_prediction_probs.keys())
    y_true_ensemble = data["labels"][all_indices]
    
    # 評価指標計算
    ensemble_f1_macro = f1_score(y_true_ensemble, ensemble_labels, average="macro")
    ensemble_f1_weighted = f1_score(y_true_ensemble, ensemble_labels, average="weighted")
    ensemble_cmi_score, ensemble_binary_f1, ensemble_macro_f1, ensemble_accuracy = calculate_cmi_score(
        ensemble_labels, y_true_ensemble
    )
    
    print(f"\n🎯 アンサンブル結果:")
    print(f"  F1 Macro: {ensemble_f1_macro:.4f}")
    print(f"  F1 Weighted: {ensemble_f1_weighted:.4f}")
    print(f"  CMI Score: {ensemble_cmi_score:.4f}")
    print(f"  Binary F1: {ensemble_binary_f1:.4f}")
    print(f"  Macro F1: {ensemble_macro_f1:.4f}")
    print(f"  Accuracy: {ensemble_accuracy:.4f}")
else:
    print("❌ 評価結果がないため、アンサンブル予測を生成できません")

## 結果の可視化

In [ ]:
# 結果の可視化
if fold_results:
    def plot_fold_comparison_fixed(fold_results: List[Dict[str, Any]], ensemble_results: Dict[str, float] = None):
        """各foldとアンサンブルの結果を比較（修正版）"""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle('Ensemble V30 モデル評価結果（修正版）', fontsize=16)
        
        # Fold別F1スコア
        fold_numbers = [r["fold"] for r in fold_results]
        f1_scores = [r["f1_macro"] for r in fold_results]
        cmi_scores = [r["cmi_score"] for r in fold_results]
        
        # F1スコア比較
        ax1.bar(fold_numbers, f1_scores, alpha=0.7, color='blue', label='Individual Folds')
        if ensemble_results:
            ax1.axhline(y=ensemble_results['f1_macro'], color='red', linestyle='--', 
                        label=f'Ensemble: {ensemble_results["f1_macro"]:.4f}', linewidth=2)
        ax1.set_xlabel('Fold')
        ax1.set_ylabel('F1 Macro Score')
        ax1.set_title('F1 Macro Score by Fold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # CMIスコア比較
        ax2.bar(fold_numbers, cmi_scores, alpha=0.7, color='green', label='Individual Folds')
        if ensemble_results:
            ax2.axhline(y=ensemble_results['cmi_score'], color='red', linestyle='--', 
                        label=f'Ensemble: {ensemble_results["cmi_score"]:.4f}', linewidth=2)
        ax2.set_xlabel('Fold')
        ax2.set_ylabel('CMI Score')
        ax2.set_title('CMI Score by Fold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 統計情報
        mean_f1 = np.mean(f1_scores)
        std_f1 = np.std(f1_scores)
        mean_cmi = np.mean(cmi_scores)
        std_cmi = np.std(cmi_scores)
        
        ax3.text(0.1, 0.8, f'Mean F1: {mean_f1:.4f}', fontsize=12, transform=ax3.transAxes)
        ax3.text(0.1, 0.7, f'Std F1: {std_f1:.4f}', fontsize=12, transform=ax3.transAxes)
        if ensemble_results:
            ax3.text(0.1, 0.6, f'Ensemble F1: {ensemble_results["f1_macro"]:.4f}', fontsize=12, transform=ax3.transAxes)
            ax3.text(0.1, 0.5, f'Improvement: {ensemble_results["f1_macro"] - mean_f1:.4f}', fontsize=12, transform=ax3.transAxes)
        ax3.set_title('F1 Score Statistics')
        ax3.axis('off')
        
        ax4.text(0.1, 0.8, f'Mean CMI: {mean_cmi:.4f}', fontsize=12, transform=ax4.transAxes)
        ax4.text(0.1, 0.7, f'Std CMI: {std_cmi:.4f}', fontsize=12, transform=ax4.transAxes)
        if ensemble_results:
            ax4.text(0.1, 0.6, f'Ensemble CMI: {ensemble_results["cmi_score"]:.4f}', fontsize=12, transform=ax4.transAxes)
            ax4.text(0.1, 0.5, f'Improvement: {ensemble_results["cmi_score"] - mean_cmi:.4f}', fontsize=12, transform=ax4.transAxes)
        ax4.set_title('CMI Score Statistics')
        ax4.axis('off')
        
        plt.tight_layout()
        plt.show()

    # アンサンブル結果を辞書にまとめる
    if 'ensemble_f1_macro' in locals():
        ensemble_results = {
            'f1_macro': ensemble_f1_macro,
            'f1_weighted': ensemble_f1_weighted,
            'cmi_score': ensemble_cmi_score,
            'binary_f1': ensemble_binary_f1,
            'macro_f1': ensemble_macro_f1,
            'accuracy': ensemble_accuracy
        }
        
        # 可視化実行
        plot_fold_comparison_fixed(fold_results, ensemble_results)
    else:
        plot_fold_comparison_fixed(fold_results)
else:
    print("❌ 評価結果がないため、可視化を実行できません")

## 結果の保存（修正版）

In [ ]:
# 結果の保存（修正版）
def save_evaluation_results_fixed(fold_results: List[Dict[str, Any]], 
                                 ensemble_results: Dict[str, float] = None,
                                 ensemble_labels: np.ndarray = None, 
                                 y_true_ensemble: np.ndarray = None):
    """評価結果を保存（修正版）"""
    print("\n💾 評価結果を保存中...")
    
    # 保存ディレクトリ作成
    save_dir = Path("../output/experiments/eval_v30_fixed")
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # 結果をまとめる
    evaluation_summary = {
        "experiment_name": EXPERIMENT_NAME,
        "trainer_name": TRAINER_NAME,
        "n_folds": N_FOLDS,
        "random_seed": RANDOM_SEED,
        "fold_results": fold_results,
        "timestamp": pd.Timestamp.now().isoformat()
    }
    
    if ensemble_results:
        evaluation_summary["ensemble_results"] = ensemble_results
    
    if ensemble_labels is not None and y_true_ensemble is not None:
        evaluation_summary["ensemble_predictions"] = ensemble_labels.tolist()
        evaluation_summary["true_labels"] = y_true_ensemble.tolist()
    
    # JSON形式で保存
    with open(save_dir / "evaluation_results.json", "w", encoding="utf-8") as f:
        json.dump(evaluation_summary, f, ensure_ascii=False, indent=2)
    
    # CSV形式で予測結果を保存
    if ensemble_labels is not None and y_true_ensemble is not None:
        all_indices = sorted(all_prediction_probs.keys())
        predictions_df = pd.DataFrame({
            "true_label": y_true_ensemble,
            "ensemble_prediction": ensemble_labels,
            "sample_index": all_indices
        })
        predictions_df.to_csv(save_dir / "ensemble_predictions.csv", index=False)
    
    # 統計サマリーを保存
    if fold_results:
        f1_scores = [r["f1_macro"] for r in fold_results]
        cmi_scores = [r["cmi_score"] for r in fold_results]
        
        summary_stats = {
            "mean_f1_folds": float(np.mean(f1_scores)),
            "std_f1_folds": float(np.std(f1_scores)),
            "mean_cmi_folds": float(np.mean(cmi_scores)),
            "std_cmi_folds": float(np.std(cmi_scores)),
        }
        
        if ensemble_results:
            summary_stats.update({
                "ensemble_f1": ensemble_results["f1_macro"],
                "ensemble_cmi": ensemble_results["cmi_score"],
                "f1_improvement": ensemble_results["f1_macro"] - np.mean(f1_scores),
                "cmi_improvement": ensemble_results["cmi_score"] - np.mean(cmi_scores)
            })
        
        with open(save_dir / "evaluation_summary.json", "w", encoding="utf-8") as f:
            json.dump(summary_stats, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 結果保存完了: {save_dir}")
    print(f"  - evaluation_results.json: 詳細結果")
    if ensemble_labels is not None:
        print(f"  - ensemble_predictions.csv: 予測結果")
    print(f"  - evaluation_summary.json: 統計サマリー")
    
    return save_dir

# 結果保存
if fold_results:
    if 'ensemble_labels' in locals() and 'y_true_ensemble' in locals():
        save_dir = save_evaluation_results_fixed(fold_results, ensemble_results, ensemble_labels, y_true_ensemble)
    else:
        save_dir = save_evaluation_results_fixed(fold_results)
else:
    print("❌ 評価結果がないため、保存をスキップします")

## 最終サマリー

In [ ]:
# 最終サマリー表示
if fold_results:
    print("\n" + "="*60)
    print("🎯 Ensemble V30 モデル評価 最終サマリー（修正版）")
    print("="*60)

    # 各foldの結果
    print("\n📊 各Foldの結果:")
    for result in fold_results:
        print(f"  Fold {result['fold']:2d}: F1={result['f1_macro']:.4f}, CMI={result['cmi_score']:.4f}")

    # 統計情報
    f1_scores = [r["f1_macro"] for r in fold_results]
    cmi_scores = [r["cmi_score"] for r in fold_results]

    print(f"\n📈 統計情報:")
    print(f"  F1 Macro - Mean: {np.mean(f1_scores):.4f}, Std: {np.std(f1_scores):.4f}")
    print(f"  CMI Score - Mean: {np.mean(cmi_scores):.4f}, Std: {np.std(cmi_scores):.4f}")

    # アンサンブル効果
    if 'ensemble_f1_macro' in locals():
        print(f"\n🎯 アンサンブル効果:")
        print(f"  F1 Macro: {ensemble_f1_macro:.4f} (改善: {ensemble_f1_macro - np.mean(f1_scores):+.4f})")
        print(f"  CMI Score: {ensemble_cmi_score:.4f} (改善: {ensemble_cmi_score - np.mean(cmi_scores):+.4f})")

    # 改善提案
    print(f"\n💡 改善提案:")
    mean_f1 = np.mean(f1_scores)
    if mean_f1 < 0.6:
        print("  - 精度が低いため、モデルアーキテクチャの見直しが必要")
        print("  - ハイパーパラメータの調整")
        print("  - 特徴量エンジニアリングの強化")
    if np.std(f1_scores) > 0.05:
        print("  - Fold間の分散が大きいため、データの安定性向上が必要")
    
    print(f"\n✅ 評価完了！結果は {save_dir} に保存されました。")
else:
    print("❌ 評価結果がないため、サマリーを表示できません")